In [1]:

import functools
import openai

def _default_retries(cls):
    original = cls.__init__

    @functools.wraps(original)
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("max_retries", 12)
        original(self, *args, **kwargs)

    cls.__init__ = __init__

for _cls in (openai.OpenAI, openai.AsyncOpenAI):
    _default_retries(_cls)


## Setup

Install the packages, then create a client. The API key comes from the `.env` file at the root of this repo, as described in the main README.

In [2]:
%pip install --quiet openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)
MODEL = "grok-4.7"

## 1. Send a request

`client.responses.create()` sends your input to Grok, and `output_text` holds the reply.

In [4]:
response = client.responses.create(
    model=MODEL,
    input="Explain in two sentences why the sky is blue.",
)
print(response.output_text)

The sky appears blue due to Rayleigh scattering, in which molecules in Earth's atmosphere scatter shorter-wavelength blue light from the sun more effectively than longer wavelengths. This scattered blue light reaches our eyes from all directions, giving the sky its characteristic color.


## 2. Give Grok instructions

Pass a list of messages instead of a string. A `system` message sets how Grok should behave for the rest of the request.

In [5]:
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": "You are a pirate. Answer in one short sentence."},
        {"role": "user", "content": "Why is the sky blue?"},
    ],
)
print(response.output_text)

Arrr, the sky be blue 'cause blue light scatters more through the air, matey!


## 3. Stream the reply

With `stream=True`, you get events while Grok writes. Printing each `response.output_text.delta` event shows the reply as it arrives.

In [6]:
stream = client.responses.create(
    model=MODEL,
    input="Write a haiku about the ocean.",
    stream=True,
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

W

aves

 crash

 on

 the

...

salt

 and

 freedom

 call

.

## 4. Hold a conversation

The API stores each response, so you don't need to resend earlier messages. Pass the previous response's `id` as `previous_response_id`, and Grok picks up where you left off.

In [7]:
first = client.responses.create(
    model=MODEL,
    input="My name is Ada and I'm learning to bake bread. Just say hi.",
)
print(first.output_text)

follow_up = client.responses.create(
    model=MODEL,
    input="What's my name, and what's a good first bread for me to try?",
    previous_response_id=first.id,
)
print(follow_up.output_text)

Hi Ada!


Your name is Ada. A good first bread to try is a simple no-knead artisan loaf—minimal kneading, just mix, rest, and bake.


If you'd rather keep track of the conversation yourself, send the full list of messages each time and set `store=False`, so nothing is kept on xAI's side.

## 5. Get structured output

Describe the shape you want with a Pydantic model and call `client.responses.parse()`. The reply comes back in `output_parsed` as an instance of your model, ready to use in code.

In [8]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    name: str
    prep_minutes: int = Field(description="Hands-on time in minutes")
    ingredients: list[str]
    steps: list[str]


response = client.responses.parse(
    model=MODEL,
    input="Give me a simple recipe for no-knead bread.",
    text_format=Recipe,
)
recipe = response.output_parsed

print(f"{recipe.name} ({recipe.prep_minutes} minutes hands-on)")
print("Ingredients:", ", ".join(recipe.ingredients))
for number, step in enumerate(recipe.steps, start=1):
    print(f"{number}. {step}")

Simple No-Knead Bread (20 minutes hands-on)
Ingredients: 3 cups all-purpose flour, 1/4 teaspoon instant yeast, 1 1/4 teaspoons salt, 1 1/2 cups water
1. Mix flour, yeast, and salt in a large bowl.
2. Add water and stir until a shaggy dough forms.
3. Cover the bowl and let the dough rise at room temperature for 12 to 18 hours.
4. Preheat oven to 450°F with a Dutch oven inside for 30 minutes.
5. Turn dough onto a floured surface and shape into a ball.
6. Place dough in the hot Dutch oven, cover, and bake 30 minutes.
7. Uncover and bake 15 minutes more until golden brown.
8. Cool on a rack before slicing.


## 6. Choose how much Grok reasons

`grok-4.7` thinks before it answers. The `reasoning` parameter sets how hard it thinks: `low`, `medium`, `high` (the default), or `xhigh`. For quick tasks that don't need reasoning, `grok-4.3` can skip it with `none`, which answers faster and costs less. `usage` shows how many output tokens went to reasoning.

In [9]:
question = "Is 1,001 a prime number? Answer yes or no, then explain in one sentence."

for model, reasoning in [(MODEL, None), ("grok-4.3", {"effort": "none"})]:
    options = {"reasoning": reasoning} if reasoning else {}
    response = client.responses.create(model=model, input=question, **options)
    usage = response.usage
    label = f"{model}, reasoning {reasoning['effort']}" if reasoning else f"{model}, default reasoning"
    print(f"{label}: {usage.output_tokens_details.reasoning_tokens} reasoning tokens, {usage.output_tokens} output tokens in total")
    print(f"  {response.output_text.strip()}\n")

grok-4.7, default reasoning: 318 reasoning tokens, 338 output tokens in total
  No, 1001 is not prime because it equals 7 × 11 × 13.



grok-4.3, reasoning none: 0 reasoning tokens, 22 output tokens in total
  No. 1,001 is not prime because it factors as 7 × 11 × 13.



## Next steps

- Let Grok call your own code in [Function Calling 101](../../function-calling/python/guide.ipynb).
- Send images to Grok in [Object Detection](../../object-detection/python/guide.ipynb).
- See everything else the API can do, like web search and X search, in the [xAI docs](https://docs.x.ai).